# Aurora 1.5: globale uurlijkse voorspelling, opgeslagen als WX-1 (v4)

Zelfstandig notebook voor Google Colab. Het controleert de GPU, draait Aurora 1.5 via Earth2Studio met de nieuwste ECMWF IFS-run als start, en bewaart de **hele wereld** (of een gekozen `--bbox`) als WX-1 (`tempest-wx1`): een map met `analysis_t0.wx1` (analyse op `t+0`), een `lead_NNN.wx1`-bestand per voorspeluur (`t+1` t/m `t+N`) plus `manifest.json`. Alle variabelen krijgen automatisch SI-eenheden (`units`) en beschrijvingen (`long_name`) uit Earth2Studio's `E2STUDIO_VOCAB`. Een regionale uitsnede (bijv. Benelux, Duitsland, Noordzee en noordelijker) kan later uit die bestanden gelezen worden, op je laptop of in een downscalingmodel.

**Wijzigingen in v4**
- Fix voor `RuntimeError: CUDAPluggableAllocator does not yet support getDeviceStats` (veilige `mem_get_info`-fallback voor VRAM-rapportage).
- Slaat ook `t+0` (`analysis_t0.wx1`) op als startveld voor latere downscaling.
- Schrijft per variabele `units`, `long_name` en `earth2studio_name` weg uit `E2STUDIO_VOCAB`.
- Directe stdout-flush (`PYTHONUNBUFFERED=1`), standaard `HOURS = 360`, `SIC_LAG_DAYS = 6`, en een snelle neerslag- en versiecheck in stap 7.

**Voor je begint**
1. Kies *Runtime > Change runtime type* en een A100 (liefst 80 GB). Aurora 1.5 heeft ongeveer 32 GB (Aurora-docs) tot 48 GB (Earth2Studio-docs) GPU-geheugen nodig. T4 en L4 zijn te klein.
2. Draai de cellen van boven naar beneden.
3. Houd de browsertab open tijdens de run.

**Opslag.** Globaal is ruim 80 variabelen x 721 x 1440 punten per uur. Bij een test met ruisdata kwam dat op ~0,3 GB per uur (het slechtste geval; echte velden comprimeren beter). Reken dus op ~15 GB voor 48 uur en ~108 GB voor 360 uur (of ~5 GB met een ruime Europese `BBOX`). Het script schrijft naar `/content` en print na het eerste uur een schatting; een cel verderop kopieert naar Drive als er genoeg ruimte is.

Controleer de licentie van de modelcheckpoints (Microsoft Aurora: MIT-licentie; neem bij commercieel gebruik contact op met Microsoft) voordat je resultaten gebruikt of deelt.

## 1. GPU controleren

In [ ]:
import sys
import torch

MIN_GB_ABORT = 30.0   # eronder stoppen
MIN_GB_WARN = 47.0    # eronder waarschuwen
FORCE = False         # True = check overslaan

if not torch.cuda.is_available():
    raise SystemExit("Geen GPU gevonden. Kies Runtime > Change runtime type > A100.")
props = torch.cuda.get_device_properties(0)
gb = props.total_memory / 1e9
print(f"GPU: {props.name}, {gb:.0f} GB")
if gb < MIN_GB_ABORT and not FORCE:
    raise SystemExit(f"Te weinig geheugen ({gb:.0f} GB) voor Aurora 1.5 globaal. Kies een A100.")
if gb < MIN_GB_WARN:
    print("Let op: de Earth2Studio-docs noemen 48 GB; bij een out-of-memory fout kies je de A100 80GB.")


## 2. Google Drive koppelen en caches instellen

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/aurora15")
HF_CACHE = DRIVE_DIR / "hf_cache"
E2S_CACHE = DRIVE_DIR / "earth2studio_cache"
for p in (DRIVE_DIR, HF_CACHE, E2S_CACHE):
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
# Officiele Earth2Studio-cachevariabele (standaard ~/.cache/earth2studio);
# optioneel apart in te stellen via EARTH2STUDIO_MODEL_CACHE en EARTH2STUDIO_DATA_CACHE.
os.environ["EARTH2STUDIO_CACHE"] = str(E2S_CACHE)
print("Caches op Drive:", DRIVE_DIR)


## 3. Pakketten installeren

In [ ]:
!pip install -q "earth2studio[aurora]" ecmwf-opendata xarray netcdf4 tempest-wx1 dask


## 4. Rekenscript schrijven

Dit is dezelfde code als `aurora15_hourly_crop.py` (v4), hier ingebed zodat je maar één bestand hoeft te uploaden.

In [ ]:
%%writefile /content/aurora15_hourly_crop.py
#!/usr/bin/env python3
"""
aurora15_hourly_crop.py (v4)

Microsoft Aurora 1.5 (0.25 graden, uurlijkse output) via NVIDIA Earth2Studio.
Het model draait globaal. Standaard wordt de hele wereld bewaard als WX-1
(tempest-wx1): een map met analysis_t0.wx1 (analyse op t+0), een lead_NNN.wx1
per voorspeluur (t+1 t/m t+N) plus manifest.json.
Met --bbox kan een uitsnede worden bewaard (ook als NetCDF met --format netcdf).

Nieuw in v4:
- Veilige VRAM-rapportage die niet crasht op PyTorch CUDAPluggableAllocator.
- Slaat ook t+0 op als analysis_t0.wx1 (nodig als begintoestand bij downscaling).
- Schrijft per variabele SI-eenheden (units) en long_name uit E2STUDIO_VOCAB.
"""
import argparse
import json
import os
import re
import time
from datetime import datetime, timedelta, timezone

import numpy as np
import torch
import xarray as xr
import zarr

from earth2studio.data import ARCO_ERA5, IFS, fetch_data
from earth2studio.data.utils import obstore_zarr_store
from earth2studio.models.px import Aurora1p5
from earth2studio.run import map_field

try:
    from earth2studio.lexicon.base import E2STUDIO_VOCAB
except Exception:
    E2STUDIO_VOCAB = {}

_PL_FALLBACK = {
    "t": ("K", "air_temperature"),
    "u": ("m s-1", "eastward_wind"),
    "v": ("m s-1", "northward_wind"),
    "z": ("m2 s-2", "geopotential"),
    "q": ("kg kg-1", "specific_humidity"),
}

_SFC_FALLBACK = {
    "t2m": ("K", "2 metre temperature"),
    "2t": ("K", "2 metre temperature"),
    "d2m": ("K", "2 metre dewpoint temperature"),
    "2d": ("K", "2 metre dewpoint temperature"),
    "skt": ("K", "skin temperature"),
    "stl1": ("K", "soil temperature level 1"),
    "u10m": ("m s-1", "10 metre U wind component"),
    "10u": ("m s-1", "10 metre U wind component"),
    "v10m": ("m s-1", "10 metre V wind component"),
    "10v": ("m s-1", "10 metre V wind component"),
    "u100m": ("m s-1", "100 metre U wind component"),
    "100u": ("m s-1", "100 metre U wind component"),
    "v100m": ("m s-1", "100 metre V wind component"),
    "100v": ("m s-1", "100 metre V wind component"),
    "i10fg": ("m s-1", "instantaneous 10 metre wind gust"),
    "msl": ("Pa", "mean sea level pressure"),
    "sp": ("Pa", "surface pressure"),
    "tcwv": ("kg m-2", "total column water vapour"),
    "tcc": ("1", "total cloud cover"),
    "lcc": ("1", "low cloud cover"),
    "mcc": ("1", "medium cloud cover"),
    "hcc": ("1", "high cloud cover"),
    "sic": ("1", "sea ice area fraction"),
    "ci": ("1", "sea ice area fraction"),
    "swvl1": ("m3 m-3", "volumetric soil water layer 1"),
    "sd": ("m", "snow depth"),
    "tp": ("m", "total precipitation"),
    "tp1h": ("m", "total precipitation (1h)"),
    "sf": ("m", "snowfall"),
    "sf1h": ("m", "snowfall (1h)"),
    "blh": ("m", "boundary layer height"),
    "ssrd": ("J m-2", "surface solar radiation downwards"),
    "ssrd1h": ("J m-2", "surface solar radiation downwards (1h)"),
    "ttr": ("J m-2", "top net thermal radiation"),
    "ttr1h": ("J m-2", "top net thermal radiation (1h)"),
    "uvb": ("J m-2", "downward UV radiation at the surface"),
    "uvb1h": ("J m-2", "downward UV radiation at the surface (1h)"),
    "lsm": ("1", "land-sea mask"),
    "z": ("m2 s-2", "surface geopotential"),
}


def format_vram_usage(device) -> str:
    """Geef VRAM-gebruik terug zonder te crashen op CUDAPluggableAllocator."""
    if device.type != "cuda":
        return "n.v.t. (CPU)"
    try:
        peak_gb = torch.cuda.max_memory_allocated(device) / 1e9
        return f"piek {peak_gb:.1f} GB"
    except RuntimeError:
        # CUDAPluggableAllocator ondersteunt getDeviceStats() niet (PyTorch #98481 / #133281)
        try:
            free_b, total_b = torch.cuda.mem_get_info(device)
            used_gb = (total_b - free_b) / 1e9
            return f"in gebruik ~{used_gb:.1f} / {total_b / 1e9:.1f} GB (via mem_get_info)"
        except Exception:
            return "onbekend (pluggable allocator)"


def variable_metadata(raw_name: str) -> dict:
    """Leid SI-eenheden en beschrijving af uit Earth2Studio E2STUDIO_VOCAB + fallbacktabel."""
    candidates = [raw_name]
    if ":" in raw_name:
        candidates.append(raw_name.replace(":sum:", ""))
        candidates.append(raw_name.split(":", 1)[0])

    desc = next((E2STUDIO_VOCAB[c] for c in candidates if c in E2STUDIO_VOCAB), "")
    units = ""
    long_name = ""
    if desc:
        m_end = re.match(r"^(.*?)\s*\(([^()]+)\)\s*$", desc)
        if m_end:
            long_name = m_end.group(1).strip()
            units = m_end.group(2).strip()
        else:
            long_name = desc.strip()
            m_mid = re.search(r"\(([^()]+)\)\s+(?:accumulated|over|since|in)\b", desc)
            if m_mid:
                units = m_mid.group(1).strip()

    if not units or not long_name:
        fb = next((_SFC_FALLBACK[c.lower()] for c in candidates if c.lower() in _SFC_FALLBACK), None)
        if fb is None:
            m_pl = re.match(r"^([tuvzq])(\d+)$", raw_name.lower())
            if m_pl:
                u_fb, ln_fb = _PL_FALLBACK[m_pl.group(1)]
                fb = (u_fb, f"{ln_fb} at {m_pl.group(2)} hPa")
        if fb is not None:
            if not units:
                units = fb[0]
            if not long_name:
                long_name = fb[1]

    meta = {"earth2studio_name": raw_name}
    if units:
        meta["units"] = units
    if long_name:
        meta["long_name"] = long_name
    return meta


def parse_init(text: str) -> datetime:
    for fmt in ("%Y-%m-%dT%H", "%Y-%m-%dT%H:%M", "%Y-%m-%d %H"):
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            pass
    return datetime.fromisoformat(text)


def latest_ifs_candidates(lookback_hours: float, max_fallbacks: int) -> list:
    """Run-tijden (UTC, naief) van nieuw naar oud: nu - lookback, naar beneden op 6 uur."""
    t = datetime.now(timezone.utc).replace(tzinfo=None) - timedelta(hours=lookback_hours)
    t = t.replace(hour=(t.hour // 6) * 6, minute=0, second=0, microsecond=0)
    return [t - timedelta(hours=6 * i) for i in range(max_fallbacks + 1)]


class ARCO_ERA5T(ARCO_ERA5):
    """ARCO-ERA5 inclusief het voorlopige ERA5T-deel.

    Earth2Studio controleert datums tegen het zarr-attribuut `valid_time_stop`
    (stabiele ERA5, loopt maanden achter). ERA5T staat in dezelfde zarr tot
    `valid_time_stop_era5t` (ongeveer 1 week achter). Hier schakelen we alleen de
    bovengrens-controle uit; de echte grens bepaalt era5t_stop().
    """

    @classmethod
    def _validate_time(cls, times):
        for t in times:
            if not (t - datetime(1900, 1, 1)).total_seconds() % 3600 == 0:
                raise ValueError(f"{t} moet op een heel uur vallen voor ARCO_ERA5T")
            if t < datetime(1940, 1, 1):
                raise ValueError(f"{t} ligt voor 1 januari 1940")


def era5t_stop():
    """Laatste tijdstip met ERA5T-data in de ARCO-zarr (naief UTC), of None."""
    store = obstore_zarr_store(f"gs://{ARCO_ERA5.ARCO_PATH.lstrip('/')}",
                               store_kwargs={"skip_signature": True})
    attrs = zarr.open(store, mode="r").attrs
    for key in ("valid_time_stop_era5t", "valid_time_stop"):
        if key in attrs:
            return datetime.fromisoformat(str(attrs[key]).rstrip("Z")), key
    return None, None


def fetch_sic(t_init, lag_days, max_back_days=10):
    """Zee-ijs (sic) uit ARCO-ERA5T, hooguit `lag_days` dagen voor t_init.

    Zee-ijs verandert traag, dus een paar dagen ouder is acceptabel. De gevraagde
    datum wordt begrensd tot de laatste ERA5T-datum; lukt een datum niet, dan
    gaat het een dag verder terug (tot max_back_days). Geeft (DataArray, datum)
    terug, of (None, None) als er niets te halen valt. Een mislukte sic mag nooit
    de keuze van de IFS-run beinvloeden: dit wordt los van die lus aangeroepen.
    """
    try:
        stop, key = era5t_stop()
    except Exception as exc:
        print(f"  Kon ARCO-metadata niet lezen ({type(exc).__name__}: {exc})")
        stop, key = None, None
    wanted = t_init - timedelta(days=lag_days)
    if stop is not None:
        print(f"  ARCO loopt tot {stop:%Y-%m-%d %H} UTC ({key})")
        wanted = min(wanted, stop)
    wanted = wanted.replace(hour=12, minute=0, second=0, microsecond=0)
    if stop is not None and wanted > stop:
        wanted = stop.replace(minute=0, second=0, microsecond=0)
    src = ARCO_ERA5T()
    for back in range(max_back_days + 1):
        t = wanted - timedelta(days=back)
        try:
            sic = fetch_data(source=src,
                             time=np.array([np.datetime64(t)], dtype="datetime64[ns]"),
                             variable=np.array(["sic"]),
                             lead_time=np.array([np.timedelta64(0, "h")]),
                             device="cpu")
            sic = sic.isel(time=0, lead_time=0, drop=True)
            if bool(sic.isnull().all()):
                raise ValueError("alleen NaN")
            print(f"  Zee-ijs (sic) uit ARCO-ERA5T van {t:%Y-%m-%d %H} UTC "
                  f"({(t_init - t).total_seconds() / 86400:.1f} dagen voor de start)")
            return sic.fillna(0.0), t
        except Exception as exc:
            print(f"  sic voor {t:%Y-%m-%d %H} UTC mislukt ({type(exc).__name__}: {exc})")
    return None, None


def attach_sic(x, sic):
    """Voeg sic (lat, lon) als variabele toe aan x (time, lead_time, variable, lat, lon)."""
    if sic is None:
        extra = xr.zeros_like(x.isel(variable=[0])).assign_coords(variable=["sic"])
    else:
        if "variable" in sic.dims:  # fetch_data laat een variabele-as van lengte 1 staan
            sic = sic.isel(variable=0, drop=True)
        sic = sic.reindex(lat=x["lat"], lon=x["lon"], method="nearest")
        sic = sic.expand_dims({"time": x["time"].values,
                               "lead_time": x["lead_time"].values})
        extra = sic.expand_dims(variable=["sic"])
    extra = extra.transpose(*x.dims).astype(x.dtype)
    return xr.concat([x, extra], dim="variable", coords="minimal", compat="override")


def fetch_initial_conditions(source, t_init, in_coords, sic=None):
    """Haal de twee invoerframes (t-6 u en t) op als xarray-DataArray.

    Earth2Studio >= 1.0: fetch_data geeft een xr.DataArray terug (geen
    (tensor, coords)-tuple meer) en het model neemt een DataArray als invoer.
    Alles wordt op de CPU opgehaald; het model verplaatst het zelf naar de GPU.
    `sic` is bij --source ifs het vooraf opgehaalde zee-ijsveld (of None = nullen).
    """
    time_arr = np.array([np.datetime64(t_init)], dtype="datetime64[ns]")
    lead = np.asarray(in_coords["lead_time"].values)
    variables_in = np.asarray(in_coords["variable"].values).astype(str)

    if source == "arco":
        x = fetch_data(source=ARCO_ERA5T(), time=time_arr, variable=variables_in,
                       lead_time=lead, device="cpu")
    else:
        # IFS open data heeft geen zee-ijs (sic): alles uit IFS, sic apart.
        x = fetch_data(source=IFS(), time=time_arr,
                       variable=variables_in[variables_in != "sic"],
                       lead_time=lead, device="cpu")
        x = attach_sic(x, sic)

    x = map_field(x, in_coords)  # variabelen en 720x1440-grid van het model kiezen

    # Een variabele die helemaal uit NaN bestaat betekent: de data bestaat (nog) niet.
    all_nan = x.isnull().all(dim=[d for d in x.dims if d in ("lat", "lon")])
    all_nan = all_nan.any(dim=[d for d in all_nan.dims if d != "variable"])
    empty = [str(v) for v in all_nan["variable"].values[all_nan.values]]
    if empty:
        raise ValueError(f"Lege velden in de invoer: {empty}")
    return x


def extract_t0_frame(x):
    """Haal het t+0 analyseframe uit invoerarray x naar (lead_time=1, variable, lat, lon)."""
    x0 = x
    for d in ("batch", "ensemble", "time"):
        if d in x0.dims:
            x0 = x0.isel({d: 0})
    if "lead_time" in x0.dims:
        lt_hours = np.asarray(x0["lead_time"].values) / np.timedelta64(1, "h")
        idx_t0 = int(np.argmin(np.abs(lt_hours)))
        x0 = x0.isel(lead_time=[idx_t0])
        x0 = x0.assign_coords(lead_time=np.array([np.timedelta64(0, "h")]))
    else:
        x0 = x0.expand_dims(lead_time=[np.timedelta64(0, "h")])
    return x0


def select_region(y, bbox):
    """y: DataArray met lead_time, variable, lat, lon. bbox=None -> hele wereld.

    Geeft een dict met float32-data (lead, variabele, lat, lon) en de assen terug.
    Globaal blijft de lengtegraad 0..359,75 (natieve Aurora-grid); bij een bbox
    wordt lon -180..180, oplopend van west naar oost.
    """
    lat = np.asarray(y["lat"].values)
    lon = np.asarray(y["lon"].values)
    if bbox is None:
        sel, lat_out, lon_out = y, lat, lon
    else:
        south, north, west, east = bbox
        lon180 = ((lon + 180.0) % 360.0) - 180.0
        lat_sel = np.where((lat >= south) & (lat <= north))[0]
        lon_sel = np.where((lon180 >= west) & (lon180 <= east))[0]
        lon_sel = lon_sel[np.argsort(lon180[lon_sel])]
        sel = y.isel(lat=lat_sel, lon=lon_sel)
        lat_out, lon_out = lat[lat_sel], lon180[lon_sel]
    sel = sel.transpose("lead_time", "variable", "lat", "lon")
    data = sel.data
    data = data.get() if hasattr(data, "get") else np.asarray(data)  # cupy -> numpy
    return {
        "data": np.ascontiguousarray(data, dtype="float32"),
        "leads": (np.asarray(sel["lead_time"].values) / np.timedelta64(1, "h")).astype(int),
        "variables": [str(v) for v in sel["variable"].values],
        "lat": lat_out,
        "lon": lon_out,
    }


def write_wx1_chunk(path, t_init, chunk, attrs, codec="zstd_shuffle"):
    """Schrijf een blok uurframes als een WX-1-container (alle variabelen, 1 bestand).

    Coordinaten gaan als 1-D variabelen met de naam van hun dimensie mee; bij het
    openen met wx1.open_dataset worden dat weer xarray-coordinaten. Tijd staat als
    int64 seconden sinds 1970-01-01 (WX-1 kent geen datetime64).
    """
    import wx1

    leads = chunk["leads"]
    secs = ((np.datetime64(t_init, "s") + leads.astype("timedelta64[h]"))
            .astype("datetime64[s]").astype("int64"))
    nlat, nlon = len(chunk["lat"]), len(chunk["lon"])
    tile = [1, min(181, nlat), min(360, nlon)]  # tegels: regio's lezen zonder alles te laden
    v = {
        "time": {"data": secs, "dimensions": ["time"],
                 "attributes": {"units": "seconds since 1970-01-01 00:00:00",
                                "calendar": "proleptic_gregorian"}},
        "lead_hours": {"data": leads.astype("int32"), "dimensions": ["time"],
                       "attributes": {"units": "hours since init_time"}},
        "lat": {"data": chunk["lat"].astype("float32"), "dimensions": ["lat"],
                "attributes": {"units": "degrees_north"}},
        "lon": {"data": chunk["lon"].astype("float32"), "dimensions": ["lon"],
                "attributes": {"units": "degrees_east"}},
    }
    for i, name in enumerate(chunk["variables"]):
        v[name.replace(":", "_")] = {
            "data": np.ascontiguousarray(chunk["data"][:, i]),
            "dimensions": ["time", "lat", "lon"],
            "chunk_shape": tile,
            "attributes": variable_metadata(name),
        }
    # 'buffered': geen fsync per bestand; de bestanden zijn opnieuw te maken.
    wx1.write_dataset(path, v, dataset_id=f"aurora15_{t_init:%Y%m%dT%H}",
                      attributes=attrs, codec=codec, workers=4, durability="buffered")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--init", default="latest",
                    help='"latest" (alleen met --source ifs) of een starttijd UTC, '
                         'bv. 2026-11-20T00 (00/06/12/18 uur)')
    ap.add_argument("--lookback-hours", type=float, default=7.0,
                    help="Bij --init latest: zoveel uur terugkijken vanaf nu")
    ap.add_argument("--max-fallbacks", type=int, default=3,
                    help="Bij --init latest: zoveel keer 6 uur eerder proberen")
    ap.add_argument("--hours", type=int, default=360, help="Aantal uren vooruit")
    ap.add_argument("--bbox", nargs=4, type=float, default=None,
                    metavar=("ZUID", "NOORD", "WEST", "OOST"),
                    help="Uitsnede in graden (west/oost mogen negatief zijn). "
                         "Weglaten = hele wereld.")
    ap.add_argument("--format", choices=["wx1", "netcdf"], default="wx1",
                    help="wx1 = een map met een WX-1-bestand per uur (standaard); "
                         "netcdf = een bestand, alleen zinvol met --bbox")
    ap.add_argument("--wx1-codec", default="zstd_shuffle",
                    choices=["zstd", "zstd_shuffle", "lz4", "raw"])
    ap.add_argument("--out", default="aurora15_{init}",
                    help="Uitvoermap (wx1) of -bestand (netcdf); {init} wordt de starttijd")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--source", choices=["arco", "ifs"], default="ifs",
                    help="ifs = ECMWF open data (actueel); arco = ERA5 (tests in het verleden)")
    ap.add_argument("--sic-lag-days", type=int, default=6,
                    help="Bij --source ifs: zee-ijs (sic) komt uit ARCO-ERA5T van zoveel dagen eerder")
    args = ap.parse_args()

    if args.init == "latest":
        if args.source != "ifs":
            ap.error('--init latest werkt alleen met --source ifs; geef bij arco een datum op')
        candidates = latest_ifs_candidates(args.lookback_hours, args.max_fallbacks)
    else:
        candidates = [parse_init(args.init)]

    device = torch.device(args.device)

    print("Model laden...")
    package = Aurora1p5.load_default_package()
    model = Aurora1p5.load_model(package).to(device).eval()
    in_coords = model.input_coords()

    sic = None
    if args.source == "ifs":
        print("Zee-ijs (sic) ophalen uit ARCO-ERA5T...")
        sic, sic_time = fetch_sic(candidates[0], args.sic_lag_days)
        if sic is None:
            print("  WAARSCHUWING: geen zee-ijs beschikbaar; ik gebruik overal 0. "
                  "Dat is veilig voor Benelux/Duitsland, maar minder goed in de Arctis.")

    x = t_init = None
    last_error = None
    for cand in candidates:
        now = datetime.now(timezone.utc).replace(tzinfo=None)
        age = (now - cand).total_seconds() / 3600
        print(f"Initiele condities ophalen ({args.source}) voor {cand:%Y-%m-%d %H} UTC "
              f"({age:.1f} uur geleden)...")
        try:
            x = fetch_initial_conditions(args.source, cand, in_coords, sic)
            t_init = cand
            break
        except Exception as exc:  # data nog niet beschikbaar of ander ophaalprobleem
            last_error = exc
            import traceback
            traceback.print_exc()
            print(f"  Niet gelukt ({type(exc).__name__}: {exc})")
            if len(candidates) > 1:
                print("  Ik probeer de run 6 uur eerder.")
    if t_init is None:
        raise RuntimeError(
            f"Geen bruikbare initiele condities gevonden in {len(candidates)} poging(en). "
            "Probeer een grotere --lookback-hours of --max-fallbacks, of controleer "
            "de netwerkverbinding en de Earth2Studio-versie.") from last_error
    print(f"Gekozen starttijd: {t_init:%Y-%m-%d %H} UTC")

    if args.format == "netcdf" and args.bbox is None and args.hours > 24:
        raise SystemExit("Globaal NetCDF met veel uren wordt enorm en past niet in het "
                         "geheugen. Gebruik --format wx1, of geef een --bbox.")

    n_chunks = -(-args.hours // 6)  # Aurora 1.5 levert per stap 6 uurframes
    out_path = args.out.replace("{init}", t_init.strftime("%Y%m%dT%H"))
    if args.format == "netcdf" and not out_path.endswith(".nc"):
        out_path += ".nc"
    attrs = {
        "source": "Microsoft Aurora 1.5 via NVIDIA Earth2Studio",
        "init_time": str(t_init),
        "initial_conditions": args.source,
        "grid": "regular lat-lon 0.25 deg",
        "region": "global" if args.bbox is None else "bbox " + " ".join(map(str, args.bbox)),
        "note": "analysis_t0.wx1 bevat de analyse op t+0 (invoervariabelen); lead_001.wx1 t/m lead_NNN.wx1 bevatten de uurlijkse voorspelstappen t+1 u t/m t+N u.",
    }
    if args.format == "wx1":
        os.makedirs(out_path, exist_ok=True)
    manifest = {"model": "Aurora 1.5", **attrs, "hours": args.hours,
                "codec": args.wx1_codec, "files": []}

    if args.format == "wx1":
        t0_chunk = select_region(extract_t0_frame(x), args.bbox)
        t0_path = os.path.join(out_path, "analysis_t0.wx1")
        write_wx1_chunk(t0_path, t_init, t0_chunk, attrs, codec=args.wx1_codec)
        t0_size = os.path.getsize(t0_path)
        manifest["analysis_file"] = "analysis_t0.wx1"
        manifest["analysis_bytes"] = t0_size
        manifest["analysis_variables"] = t0_chunk["variables"]
        manifest.update(
            lat=[float(t0_chunk["lat"][0]), float(t0_chunk["lat"][-1]), len(t0_chunk["lat"])],
            lon=[float(t0_chunk["lon"][0]), float(t0_chunk["lon"][-1]), len(t0_chunk["lon"])],
        )
        with open(os.path.join(out_path, "manifest.json"), "w") as fh:
            json.dump(manifest, fh, indent=1)
        print(f"  Analyse (t+0): {t0_size / 1e6:.0f} MB -> analysis_t0.wx1 "
              f"({len(t0_chunk['variables'])} variabelen)")
        del t0_chunk

    mem_chunks = []   # alleen voor --format netcdf
    t_start = time.time()
    try:
        with torch.inference_mode():
            for k, y in enumerate(model.create_iterator(x)):
                # y: (batch?, time, lead_time=6 uurframes, variable, lat, lon)
                for d in ("batch", "ensemble"):
                    if d in y.dims:
                        y = y.isel({d: 0})
                if "time" in y.dims:
                    y = y.isel(time=0)
                # Per uur naar het geheugen halen en wegschrijven: houdt de RAM-piek klein
                # (globaal ~0,35 GB per uur ruw in plaats van ~2 GB per 6-uursblok).
                for j in range(y.sizes["lead_time"]):
                    lead_h = int(np.asarray(y["lead_time"].values[j]) / np.timedelta64(1, "h"))
                    if lead_h > args.hours:
                        break
                    chunk = select_region(y.isel(lead_time=[j]), args.bbox)
                    if k == 0 and j == 0 and device.type == "cuda":
                        print(f"VRAM na eerste stap: {format_vram_usage(device)}")
                    if args.format == "wx1":
                        fname = f"lead_{lead_h:03d}.wx1"
                        write_wx1_chunk(os.path.join(out_path, fname), t_init, chunk, attrs,
                                        codec=args.wx1_codec)
                        size = os.path.getsize(os.path.join(out_path, fname))
                        manifest["files"].append({"file": fname, "lead_hours": lead_h,
                                                  "bytes": size})
                        manifest.update(variables=chunk["variables"],
                                        lat=[float(chunk["lat"][0]), float(chunk["lat"][-1]),
                                             len(chunk["lat"])],
                                        lon=[float(chunk["lon"][0]), float(chunk["lon"][-1]),
                                             len(chunk["lon"])])
                        if k == 0 and j == 0:
                            print(f"  Eerste uur: {size / 1e6:.0f} MB; verwacht totaal "
                                  f"~{size * args.hours / 1e9:.0f} GB voor {args.hours} u")
                    else:
                        mem_chunks.append(chunk)
                    del chunk
                if args.format == "wx1":
                    with open(os.path.join(out_path, "manifest.json"), "w") as fh:
                        json.dump(manifest, fh, indent=1)  # na elk blok: een afgebroken run blijft bruikbaar
                print(f"  t+{lead_h:>3d} u klaar  ({time.time() - t_start:.0f} s)")
                if k + 1 >= n_chunks:
                    break
    except torch.cuda.OutOfMemoryError:
        print("Onvoldoende GPU-geheugen. Aurora 1.5 globaal heeft ~32-48 GB nodig; "
              "draai dit op een grotere GPU.")
        raise

    if args.format == "netcdf":
        variables = mem_chunks[0]["variables"]
        arr = np.concatenate([c["data"] for c in mem_chunks], axis=0)
        leads = np.concatenate([c["leads"] for c in mem_chunks])
        valid = np.datetime64(t_init, "ns") + leads.astype("timedelta64[h]")
        ds = xr.Dataset(
            # ':' in namen zoals tp:sum:1h kan in NetCDF lastig zijn: vervang door '_'
            {v.replace(":", "_"): (("time", "lat", "lon"), arr[:, i])
             for i, v in enumerate(variables)},
            coords={"time": valid, "lat": mem_chunks[0]["lat"], "lon": mem_chunks[0]["lon"]},
            attrs=attrs,
        )
        for v_name in variables:
            ds[v_name.replace(":", "_")].attrs.update(variable_metadata(v_name))
        enc = {v: {"zlib": True, "complevel": 4, "dtype": "float32"} for v in ds.data_vars}
        ds.to_netcdf(out_path, encoding=enc)
        n_steps, n_vars = len(leads), len(variables)
    else:
        n_steps = len(manifest["files"])
        n_vars = len(manifest.get("variables", []))
        total = (sum(f["bytes"] for f in manifest["files"]) + manifest.get("analysis_bytes", 0)) / 1e9
        print(f"Totaal op schijf (incl. t+0): {total:.1f} GB")

    if device.type == "cuda":
        print(f"VRAM aan eind van run: {format_vram_usage(device)}")
    print(f"Klaar: {out_path}  ({n_steps} uurstappen, {n_vars} variabelen)")
    print(f"OUTPUT_FILE={out_path}")


if __name__ == "__main__":
    main()


## 5. Parameters en run

- `INIT = "latest"`: het script zoekt zelf de nieuwste IFS-run, rondt af op 00/06/12/18 UTC en probeert bij mislukken automatisch 6 uur eerder. Je kunt ook een vaste starttijd opgeven, bv. `"2026-11-20T00"`.
- `BBOX = None`: de hele wereld (~108 GB voor 360 uur). Geef `[zuid, noord, west, oost]` op voor een uitsnede (bijv. `[32.0, 70.0, -25.0, 40.0]` voor Europa + ruime downscalingbuffer, ~5 GB voor 360 uur).
- `FORMAT = "wx1"`: een map met `analysis_t0.wx1` (t+0) en een `lead_NNN.wx1` per uur. `"netcdf"` kan alleen met een `BBOX`.
- `SOURCE = "ifs"`: zee-ijs ontbreekt in open data en wordt aangevuld met ARCO-ERA5T (voorlopige ERA5, ~1 week achter), begrensd op de laatste beschikbare datum. Lukt dat niet, dan wordt overal 0 gebruikt, met een waarschuwing. Niet gevalideerd.
- `SOURCE = "arco"`: ERA5 + ERA5T, alleen voor tests met een vaste datum van minstens ~1 week geleden.

In [ ]:
import shutil
import subprocess

INIT = "latest"            # "latest" of een vaste starttijd, bv. "2026-11-20T00"
LOOKBACK_HOURS = 7         # bij "latest": zoveel uur terugkijken vanaf nu
MAX_FALLBACKS = 3          # en zo vaak 6 uur eerder proberen
HOURS = 360
SOURCE = "ifs"             # "ifs" of "arco" (arco vereist een vaste INIT)
SIC_LAG_DAYS = 6
BBOX = None                # None = hele wereld, of [zuid, noord, west, oost]
# BBOX = [32.0, 70.0, -25.0, 40.0]  # ~5 GB voor 360u i.p.v. ~108 GB globaal
FORMAT = "wx1"             # "wx1" of "netcdf" (netcdf alleen met BBOX)

est_gb = 0.3 * HOURS if BBOX is None else 0.02 * HOURS
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Schatting opslag ~{est_gb:.0f} GB (slechtste geval), vrij op /content: {free_gb:.0f} GB")
if free_gb < est_gb:
    raise SystemExit("Te weinig schijfruimte op /content voor deze run; verlaag HOURS of geef een BBOX.")

cmd = [
    sys.executable, "/content/aurora15_hourly_crop.py",
    "--init", INIT,
    "--lookback-hours", str(LOOKBACK_HOURS),
    "--max-fallbacks", str(MAX_FALLBACKS),
    "--hours", str(HOURS),
    "--source", SOURCE, "--sic-lag-days", str(SIC_LAG_DAYS),
    "--format", FORMAT,
    "--out", "/content/aurora15_{init}", "--device", "cuda",
]
if BBOX is not None:
    cmd += ["--bbox", *[str(v) for v in BBOX]]
print(" ".join(cmd))

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, env=env)
OUT = None
for line in proc.stdout:
    print(line, end="")
    if line.startswith("OUTPUT_FILE="):
        OUT = line.strip().split("=", 1)[1]
proc.wait()
if proc.returncode != 0 or OUT is None:
    raise RuntimeError("De run is mislukt; zie de uitvoer hierboven.")


## 6. Resultaat naar Drive kopiëren

Alleen als het past; Drive heeft standaard 15 GB.

In [ ]:
import shutil

src = Path(OUT)
files = [p for p in src.rglob("*") if p.is_file()] if src.is_dir() else [src]
size_gb = sum(p.stat().st_size for p in files) / 1e9
free_gb = shutil.disk_usage(DRIVE_DIR).free / 1e9
print(f"Resultaat: {size_gb:.1f} GB, vrij op Drive: {free_gb:.1f} GB")
if size_gb < 0.95 * free_gb:
    dest = DRIVE_DIR / src.name
    if src.is_dir():
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dest)
    print("Bewaard op Drive:", dest)
else:
    print("Past niet op Drive. De bestanden staan nog in", src,
          "- download ze (bv. met een gedeelde schijf of rclone) voordat de sessie afloopt.")


## 7. Snelle controle van de uitvoer

De bestanden worden lazy geopend (dask): alleen wat je selecteert wordt gelezen. De tijd staat in WX-1 als seconden sinds 1970 en wordt hier omgezet. Ook worden `analysis_t0.wx1`, opgeslagen eenheden en een neerslagcheck getoond.

In [ ]:
import glob
from importlib.metadata import version as pkg_version
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import wx1

try:
    import earth2studio
    e2s_ver = getattr(earth2studio, "__version__", None) or pkg_version("earth2studio")
except Exception:
    e2s_ver = "onbekend"
print(f"earth2studio: {e2s_ver} | tempest-wx1: {getattr(wx1, '__version__', pkg_version('tempest-wx1'))}")

out_dir = Path(OUT)
if out_dir.is_dir():
    t0_file = out_dir / "analysis_t0.wx1"
    if t0_file.is_file():
        ds_t0 = wx1.open_dataset(t0_file)
        t0_vars = [v for v in ds_t0.data_vars if v != "lead_hours"]
        print(f"Analyse t+0 (analysis_t0.wx1): {len(t0_vars)} variabelen")
        ds_t0.close()
    else:
        print("Let op: geen analysis_t0.wx1 gevonden in", out_dir)

    from wx1.xarray_backend import Wx1BackendEntrypoint
    files = sorted(glob.glob(f"{OUT}/lead_*.wx1"))
    ds = xr.open_mfdataset(files, engine=Wx1BackendEntrypoint, combine="nested",
                           concat_dim="time", data_vars="minimal", coords="minimal",
                           compat="override", chunks={"time": 1})
    ds = ds.assign_coords(time=ds["time"].values.astype("datetime64[s]"))
else:
    ds = xr.open_dataset(OUT)
print(ds)

names = [v for v in ds.data_vars if v != "lead_hours"]
pick = next((v for v in names if v.lower() in ("t2m", "2t")), names[0])
print(f"Grafiek van: {pick} | attrs: {ds[pick].attrs}")

tp_var = next((v for v in names if v.lower().startswith("tp")), None)
if tp_var is not None:
    tp_step0 = ds[tp_var].isel(time=0).load()
    tp_max = float(np.nanmax(tp_step0.values))
    print(f"Neerslagcheck ({tp_var} op t+1u): max = {tp_max:.6f} ({tp_step0.attrs.get('units', '?')}) -> verwacht ~0.001-0.05 m (niet ~1e-6 m)")

last = ds[pick].isel(time=-1).load()
fig, ax = plt.subplots(figsize=(9, 5))
last.plot(ax=ax)
ax.set_title(f"{pick} op {str(ds.time.values[-1])[:16]} UTC")
plt.show()


## 8. Later: regio uitsnijden voor downscaling

Voorbeeld voor Benelux, Duitsland, Noordzee en noordelijker. Dit leest alleen de tegels die de regio raken, dus het werkt ook op je laptop met de hele run. Let op: de globale grid heeft lengtegraad 0 tot 359,75; west van Greenwich is dus 348 tot 360. De bestanden bevatten nu per variabele `units`, `long_name` en `earth2studio_name` uit `E2STUDIO_VOCAB`, en `analysis_t0.wx1` bevat de `t+0`-analyse om een downscalingmodel te initialiseren.

In [ ]:
# Regio: Benelux + Duitsland + Noordzee + noordelijker (graden)
south, north, west, east = 46.0, 66.0, -12.0, 25.0
lon360 = lambda x: x % 360
sub = xr.concat(
    [ds.sel(lat=slice(north, south), lon=slice(lon360(west), 360)),
     ds.sel(lat=slice(north, south), lon=slice(0, east))],
    dim="lon", data_vars="minimal", coords="minimal", compat="override",
)
sub = sub.assign_coords(lon=((sub["lon"] + 180) % 360) - 180).sortby("lon")
print(dict(sub.sizes))
# sub.to_netcdf("aurora15_regio.nc")   # of verder met je eigen downscaler
